# Credit Default Risk Modeling - Exploratory Data Analysis (EDA)

**Objective**:
- Inspect data quality, missing values, and distributions.
- Eliminate **Data Leakage** (e.g., `FPD10+`, post-disbursement delinquency metrics) and uninformative **PII Identifiers** (e.g., `customer_id`).
- Analyze default rate (**Bad Rate %**) across primary target `DPD10_3MOB` (Days Past Due >= 10 in 3 Months on Book).
- Evaluate risk patterns across demographics, geographic regions (`province`), device OS (`operating_system`), loan terms, and extracted **CIC Bureau features**.

In [ ]:
# ==============================================================================
# 1. Environment & Library Setup
# ==============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

# Configure modern plot aesthetics
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

# Color palettes for risk visualizations
PRIMARY_COLOR = "#2b5c8f"  # Navy Blue (Volume / Safe)
RISK_COLOR = "#e74c3c"     # Coral Red (Default / High Risk)
PALETTE_BINARY = ["#2b5c8f", "#e74c3c"]
PALETTE_CATEGORICAL = ["#2b5c8f", "#3498db", "#f39c12", "#e74c3c", "#9b59b6", "#1abc9c"]

In [ ]:
# ==============================================================================
# 2. Load Extracted Dataset
# ==============================================================================
data_path = "../data/processed/data_extracted.parquet"

if not os.path.exists(data_path):
    # Fallback to local working directory path if running from root
    data_path = "data/processed/data_extracted.parquet"

df = pd.read_parquet(data_path)
print(f"Loaded dataset successfully: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(5)

In [ ]:
# ==============================================================================
# 3. Audit & Drop Data Leakage & Identifier Features
# ==============================================================================
target_col = "DPD10_3MOB"

# Explicitly identify leakage (post-disbursement performance) and ID columns
leakage_keywords = ["fpd", "collection", "recovery", "writeoff", "chargeoff", "overdue_days"]
id_keywords = ["customer_id", "application_id", "contract_id", "cif", "id_number", "id_card", "phone_number", "email", "full_name"]

dropped_cols = []
for col in df.columns:
    if col == target_col:
        continue
    col_l = col.lower().strip()
    if any(k in col_l for k in leakage_keywords) or (("dpd" in col_l) and (col != target_col)):
        dropped_cols.append(col)
    elif any(k in col_l for k in id_keywords) or col_l in ["id", "cif_no", "cust_id", "app_id"]:
        dropped_cols.append(col)
    elif df[col].nunique(dropna=False) <= 1:  # Constant / Zero variance
        dropped_cols.append(col)

if dropped_cols:
    print(f"Dropped {len(dropped_cols)} leakage/identifier/constant columns:")
    for col in dropped_cols:
        print(f"  - {col}")
    df = df.drop(columns=dropped_cols)
else:
    print("No additional leakage or identifier columns found.")

print(f"\nRemaining clean features for EDA: {df.shape[1]} columns ({df.shape[0]:,} records)")

In [ ]:
# ==============================================================================
# 4. Missing Value Analysis
# ==============================================================================
missing_counts = df.isna().sum()
missing_rates = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Rate (%)": missing_rates
})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing Rate (%)", ascending=False)

print(f"Total features with missing values: {len(missing_df)}")

if not missing_df.empty:
    plt.figure(figsize=(10, max(4, len(missing_df) * 0.35)))
    sns.barplot(
        x=missing_df["Missing Rate (%)"][:20],
        y=missing_df.index[:20],
        palette="Blues_r"
    )
    plt.title("Top Missing Value Features (%)", fontsize=13, fontweight="bold", pad=12)
    plt.xlabel("Missing Rate (%)", fontsize=10)
    plt.xlim(0, 100)
    
    # Annotate percentage on bars
    for i, v in enumerate(missing_df["Missing Rate (%)"][:20]):
        plt.text(v + 1, i, f"{v:.1f}%", va="center", fontsize=9, fontweight="bold", color="#333333")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# 5. Target Variable Analysis (DPD10_3MOB - Default / Bad Rate)
# ==============================================================================
if target_col in df.columns:
    target_counts = df[target_col].value_counts().sort_index()
    bad_rate = df[target_col].mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # 1. Donut Chart
    labels = ["Non-Default (0)", "Default (1)"]
    axes[0].pie(
        target_counts,
        labels=labels,
        autopct="%1.2f%%",
        startangle=90,
        colors=PALETTE_BINARY,
        explode=(0, 0.08),
        wedgeprops=dict(width=0.4, edgecolor="white", linewidth=2)
    )
    axes[0].set_title(f"Overall Portfolio Bad Rate: {bad_rate:.2f}%", fontsize=13, fontweight="bold")

    # 2. Count Bar Chart with exact volume labels
    sns.barplot(x=labels, y=target_counts.values, palette=PALETTE_BINARY, ax=axes[1])
    for p in axes[1].patches:
        height = int(p.get_height())
        axes[1].annotate(
            f"{height:,} ({height/len(df)*100:.1f}%)",
            (p.get_x() + p.get_width() / 2., p.get_height() / 2),
            ha="center", va="center", color="white", fontweight="bold", fontsize=11
        )
    axes[1].set_title("Loan Count by Target Status", fontsize=13, fontweight="bold")
    axes[1].set_ylabel("Number of Applications", fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print(f"Target column '{target_col}' not found in DataFrame.")

In [ ]:
# ==============================================================================
# 6. Helper Function: Dual-Axis Bivariate Plot (Volume vs Bad Rate %)
# ==============================================================================
def plot_bivariate_dual_axis(data, col, target="DPD10_3MOB", title=None, min_count=10):
    """
    Creates a standard Risk Analytics dual-axis chart:
    - Bar (Left Axis): Application Volume
    - Line (Right Axis): Bad Rate %
    """
    if col not in data.columns or target not in data.columns:
        return

    stats = data.groupby(col).agg(
        volume=(target, "count"),
        bad_rate=(target, "mean")
    ).reset_index()

    # Filter out categories with very few samples to prevent visual skew
    stats = stats[stats["volume"] >= min_count].copy()
    stats["bad_rate_pct"] = stats["bad_rate"] * 100
    stats = stats.sort_values("volume", ascending=False)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    # Left axis: Application Volume (Bar)
    sns.barplot(data=stats, x=col, y="volume", ax=ax1, color=PRIMARY_COLOR, alpha=0.85)
    ax1.set_ylabel("Application Volume (Count)", color=PRIMARY_COLOR, fontsize=10, fontweight="bold")
    ax1.grid(False)

    # Right axis: Bad Rate % (Line)
    sns.lineplot(data=stats, x=col, y="bad_rate_pct", ax=ax2, color=RISK_COLOR, marker="o", linewidth=2.5, markersize=8)
    ax2.set_ylabel("Bad Rate (%)", color=RISK_COLOR, fontsize=10, fontweight="bold")
    ax2.set_ylim(0, max(stats["bad_rate_pct"].max() * 1.3, 5))

    # Annotate Bad Rate % on markers
    for i, row in stats.reset_index().iterrows():
        ax2.annotate(
            f"{row['bad_rate_pct']:.1f}%",
            (i, row["bad_rate_pct"] + 0.4),
            ha="center", color=RISK_COLOR, fontweight="bold", fontsize=9
        )

    plt.title(title or f"Risk Analysis: {col} vs Bad Rate (%)", fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# 7. Geographic & Demographic Analysis (Province & Operating System)
# ==============================================================================
# 1. Geographic Region (Hanoi, Danang, Saigon, Others)
if "province" in df.columns:
    plot_bivariate_dual_axis(df, "province", title="Geographic Distribution: Province vs Bad Rate (%)")

# 2. Operating System (iOS, Android, Unknown)
if "operating_system" in df.columns:
    plot_bivariate_dual_axis(df, "operating_system", title="Device Risk: Operating System vs Bad Rate (%)")

In [ ]:
# ==============================================================================
# 8. CIC Credit Bureau Feature Analysis
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Has Finance Company Loan (cic_has_fin_loan)
if "cic_has_fin_loan" in df.columns:
    fin_stats = df.groupby("cic_has_fin_loan")[target_col].agg(["count", "mean"]).reset_index()
    fin_stats["label"] = fin_stats["cic_has_fin_loan"].map({0: "No Finance Loan (0)", 1: "Has Finance Loan (1)"})
    sns.barplot(data=fin_stats, x="label", y=fin_stats["mean"] * 100, ax=axes[0, 0], palette=PALETTE_BINARY)
    axes[0, 0].set_title("Finance Company Loan Exposure vs Bad Rate (%)", fontweight="bold")
    axes[0, 0].set_ylabel("Bad Rate (%)")
    for p in axes[0, 0].patches:
        axes[0, 0].annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.3), ha="center", fontweight="bold")

# 2. Primary Institution Type (cic_primary_inst_type)
if "cic_primary_inst_type" in df.columns:
    inst_stats = df.groupby("cic_primary_inst_type")[target_col].agg(["count", "mean"]).reset_index()
    inst_stats = inst_stats.sort_values("mean", ascending=False)
    sns.barplot(data=inst_stats, x="cic_primary_inst_type", y=inst_stats["mean"] * 100, ax=axes[0, 1], palette="Blues_r")
    axes[0, 1].set_title("Primary Debt Institution vs Bad Rate (%)", fontweight="bold")
    axes[0, 1].set_ylabel("Bad Rate (%)")
    for p in axes[0, 1].patches:
        axes[0, 1].annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.3), ha="center", fontweight="bold")

# 3. CIC Credit Score Distribution (Non-Default vs Default)
if "cic_credit_score" in df.columns:
    score_valid = df[df["cic_credit_score"].notna()]
    if not score_valid.empty:
        sns.kdeplot(data=score_valid[score_valid[target_col] == 0]["cic_credit_score"], label="Non-Default (0)", color=PRIMARY_COLOR, ax=axes[1, 0], fill=True, alpha=0.3)
        sns.kdeplot(data=score_valid[score_valid[target_col] == 1]["cic_credit_score"], label="Default (1)", color=RISK_COLOR, ax=axes[1, 0], fill=True, alpha=0.3)
        axes[1, 0].set_title("CIC Credit Score Distribution by Target", fontweight="bold")
        axes[1, 0].legend()

# 4. Recent Inquiries in Last 90 Days (cic_inquiry_3m)
if "cic_inquiry_3m" in df.columns:
    inq_stats = df.groupby("cic_inquiry_3m")[target_col].agg(["count", "mean"]).reset_index()
    inq_stats = inq_stats[inq_stats["count"] >= 20]
    sns.barplot(data=inq_stats, x="cic_inquiry_3m", y=inq_stats["mean"] * 100, ax=axes[1, 1], palette="Reds")
    axes[1, 1].set_title("CIC Inquiries (Last 3M) vs Bad Rate (%)", fontweight="bold")
    axes[1, 1].set_ylabel("Bad Rate (%)")
    for p in axes[1, 1].patches:
        axes[1, 1].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.3), ha="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 9. Correlation Analysis & Top Risk Feature Ranking (Clean Features Only)
# ==============================================================================
numeric_df = df.select_dtypes(include=[np.number])

if target_col in numeric_df.columns:
    # Compute Spearman rank correlation with target
    corr_target = numeric_df.corr(method="spearman")[target_col].drop(target_col).dropna().sort_values(ascending=False)

    # Select top positive and negative risk features
    top_risk_features = pd.concat([corr_target.head(8), corr_target.tail(8)]).drop_duplicates()

    # 1. Bar plot: Top Feature Correlations
    plt.figure(figsize=(10, 6))
    colors = [RISK_COLOR if x > 0 else PRIMARY_COLOR for x in top_risk_features.values]
    sns.barplot(x=top_risk_features.values, y=top_risk_features.index, palette=colors)
    plt.title("Top Feature Correlations with Default Risk (DPD10_3MOB)", fontsize=13, fontweight="bold", pad=12)
    plt.xlabel("Spearman Rank Correlation Coefficient", fontsize=10)
    plt.axvline(0, color="#666666", linestyle="--", linewidth=0.8)
    
    for i, v in enumerate(top_risk_features.values):
        offset = 0.005 if v >= 0 else -0.018
        plt.text(v + offset, i, f"{v:+.3f}", va="center", fontsize=9, fontweight="bold")
    
    plt.tight_layout()
    plt.show()

    # 2. Triangular Correlation Heatmap of top 12 predictor features
    top_cols = corr_target.abs().sort_values(ascending=False).head(12).index.tolist() + [target_col]
    corr_matrix = numeric_df[top_cols].corr(method="spearman")
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    plt.figure(figsize=(11, 8))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, linewidths=0.5)
    plt.title("Correlation Heatmap: Top Predictor Features & Target", fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.show()

## Key Takeaways & Preprocessing Action Plan:

1. **Leakage Elimination**: All post-disbursement performance proxies (e.g., `FPD10+`, other DPD targets) and arbitrary ID identifiers (`customer_id`, `contract_id`) were filtered out to prevent data leakage and overfitting.
2. **Class Imbalance**: The dataset contains an imbalanced target distribution (`DPD10_3MOB`). Stratified K-Fold / train-test split and appropriate evaluation metrics (AUC-ROC, Gini, Precision-Recall AUC) are required.
3. **CIC Bureau Signals**: Features like `cic_has_fin_loan`, `cic_primary_inst_type`, and `cic_inquiry_3m` provide strong risk divergence.
4. **Categorical Features**: `province` (`Hanoi`, `Danang`, `Saigon`, `Others`) and `operating_system` (`iOS`, `Android`, `Unknown`) should be one-hot encoded or target encoded.
5. **Missing Values**: CIC scores and credit limits have missing values for thin-file / new-to-credit customers. Missing indicators and median/zero imputation strategies should be set up in `src/components/preprocessor.py`.